In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold

def generate_kaggle_submission():
    # Kaggle default path for input datasets
    base_dir = '/kaggle/input/competitions/march-machine-learning-mania-2026/'
    
    # If running locally for testing, fallback to local path if Kaggle path doesn't exist
    if not os.path.exists(base_dir):
        base_dir = './'
    
    m_reg = pd.read_csv(os.path.join(base_dir, 'MRegularSeasonDetailedResults.csv'))
    w_reg = pd.read_csv(os.path.join(base_dir, 'WRegularSeasonDetailedResults.csv'))
    m_tourney = pd.read_csv(os.path.join(base_dir, 'MNCAATourneyCompactResults.csv'))
    w_tourney = pd.read_csv(os.path.join(base_dir, 'WNCAATourneyCompactResults.csv'))
    m_seeds = pd.read_csv(os.path.join(base_dir, 'MNCAATourneySeeds.csv'))
    w_seeds = pd.read_csv(os.path.join(base_dir, 'WNCAATourneySeeds.csv'))

    m_tourney = m_tourney[m_tourney['Season'] >= 2003].copy()
    w_tourney = w_tourney[w_tourney['Season'] >= 2010].copy()

    def parse_seed(seed_val):
        return int(seed_val[1:3])

    m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)
    w_seeds['SeedNum'] = w_seeds['Seed'].apply(parse_seed)

    def extract_metrics(df):
        w_cols = ['WScore', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF']
        l_cols = ['LScore', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']
        
        base_cols = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
        
        wins_df = df[['Season', 'WTeamID'] + w_cols].copy()
        wins_df.columns = ['Season', 'TeamID'] + base_cols
        wins_df['wins'] = 1
        wins_df['losses'] = 0
        
        losses_df = df[['Season', 'LTeamID'] + l_cols].copy()
        losses_df.columns = ['Season', 'TeamID'] + base_cols
        losses_df['wins'] = 0
        losses_df['losses'] = 1
        
        combined = pd.concat([wins_df, losses_df], ignore_index=True)
        grouped = combined.groupby(['Season', 'TeamID']).sum().reset_index()
        
        grouped['total_games'] = grouped['wins'] + grouped['losses']
        grouped['win_rate'] = grouped['wins'] / grouped['total_games']
        
        out_cols = ['Season', 'TeamID', 'win_rate']
        for col in base_cols:
            grouped[f'avg_{col}'] = grouped[col] / grouped['total_games']
            out_cols.append(f'avg_{col}')
            
        grouped['fg_pct'] = grouped['FGM'] / grouped['FGA'].replace(0, 1)
        grouped['fg3_pct'] = grouped['FGM3'] / grouped['FGA3'].replace(0, 1)
        grouped['ft_pct'] = grouped['FTM'] / grouped['FTA'].replace(0, 1)
        grouped['ast_to_ratio'] = grouped['Ast'] / grouped['TO'].replace(0, 1)
        
        out_cols.extend(['fg_pct', 'fg3_pct', 'ft_pct', 'ast_to_ratio'])
        return grouped[out_cols]

    m_metrics = extract_metrics(m_reg)
    w_metrics = extract_metrics(w_reg)

    def compile_features(tourney_data, metrics_data, seeds_data):
        wins = tourney_data[['Season', 'WTeamID', 'LTeamID']].copy()
        wins.rename(columns={'WTeamID': 'Team1', 'LTeamID': 'Team2'}, inplace=True)
        wins['target'] = 1.0
        
        losses = tourney_data[['Season', 'LTeamID', 'WTeamID']].copy()
        losses.rename(columns={'LTeamID': 'Team1', 'WTeamID': 'Team2'}, inplace=True)
        losses['target'] = 0.0
        
        df = pd.concat([wins, losses], ignore_index=True)
        
        df = pd.merge(df, seeds_data, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left')
        df.rename(columns={'SeedNum': 'T1_Seed'}, inplace=True)
        df.drop(['TeamID', 'Seed'], axis=1, inplace=True, errors='ignore')
        
        df = pd.merge(df, seeds_data, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], how='left')
        df.rename(columns={'SeedNum': 'T2_Seed'}, inplace=True)
        df.drop(['TeamID', 'Seed'], axis=1, inplace=True, errors='ignore')
        
        df['T1_Seed'] = df['T1_Seed'].fillna(10)
        df['T2_Seed'] = df['T2_Seed'].fillna(10)
        df['Seed_diff'] = df['T1_Seed'] - df['T2_Seed']
        
        metrics_cols = [c for c in metrics_data.columns if c not in ['Season', 'TeamID']]
        
        df = pd.merge(df, metrics_data, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left')
        df.drop('TeamID', axis=1, inplace=True)
        df.rename(columns={c: f'T1_{c}' for c in metrics_cols}, inplace=True)
        
        df = pd.merge(df, metrics_data, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], how='left')
        df.drop('TeamID', axis=1, inplace=True)
        df.rename(columns={c: f'T2_{c}' for c in metrics_cols}, inplace=True)
        
        for c in metrics_cols:
            df[f'{c}_diff'] = df[f'T1_{c}'] - df[f'T2_{c}']
            
        return df

    m_train = compile_features(m_tourney, m_metrics, m_seeds)
    w_train = compile_features(w_tourney, w_metrics, w_seeds)
    full_train = pd.concat([m_train, w_train], ignore_index=True).dropna(subset=['target'])

    exclude_cols = ['Season', 'Team1', 'Team2', 'target']
    features = [c for c in full_train.columns if c not in exclude_cols]

    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    models = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(full_train)):
        x_tr = full_train.iloc[train_idx][features]
        y_tr = full_train.iloc[train_idx]['target']
        x_va = full_train.iloc[val_idx][features]
        y_va = full_train.iloc[val_idx]['target']
        
        tr_ds = lgb.Dataset(x_tr, label=y_tr)
        va_ds = lgb.Dataset(x_va, label=y_va)
        
        params = {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'learning_rate': 0.03,
            'num_leaves': 15,
            'max_depth': 4,
            'feature_fraction': 0.7,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'seed': 2026 + fold_idx,
            'verbose': -1
        }
        
        model = lgb.train(
            params,
            tr_ds,
            num_boost_round=1500,
            valid_sets=[tr_ds, va_ds],
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        models.append(model)
        
    sub_df = pd.read_csv(os.path.join(base_dir, 'SampleSubmissionStage1.csv'))
    sub_df['Season'] = sub_df['ID'].apply(lambda x: int(x.split('_')[0]))
    sub_df['Team1'] = sub_df['ID'].apply(lambda x: int(x.split('_')[1]))
    sub_df['Team2'] = sub_df['ID'].apply(lambda x: int(x.split('_')[2]))
    
    m_sub = sub_df[sub_df['Team1'] < 3000].copy()
    w_sub = sub_df[sub_df['Team1'] >= 3000].copy()
    
    def process_test(df, metrics_data, seeds_data):
        df = pd.merge(df, seeds_data, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left')
        df.rename(columns={'SeedNum': 'T1_Seed'}, inplace=True)
        df.drop(['TeamID', 'Seed'], axis=1, inplace=True, errors='ignore')
        
        df = pd.merge(df, seeds_data, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], how='left')
        df.rename(columns={'SeedNum': 'T2_Seed'}, inplace=True)
        df.drop(['TeamID', 'Seed'], axis=1, inplace=True, errors='ignore')
        
        df['T1_Seed'] = df['T1_Seed'].fillna(10)
        df['T2_Seed'] = df['T2_Seed'].fillna(10)
        df['Seed_diff'] = df['T1_Seed'] - df['T2_Seed']
        
        metrics_cols = [c for c in metrics_data.columns if c not in ['Season', 'TeamID']]
        
        df = pd.merge(df, metrics_data, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left')
        df.drop('TeamID', axis=1, inplace=True)
        df.rename(columns={c: f'T1_{c}' for c in metrics_cols}, inplace=True)
        
        df = pd.merge(df, metrics_data, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], how='left')
        df.drop('TeamID', axis=1, inplace=True)
        df.rename(columns={c: f'T2_{c}' for c in metrics_cols}, inplace=True)
        
        for c in metrics_cols:
            df[f'{c}_diff'] = df[f'T1_{c}'] - df[f'T2_{c}']
            
        return df

    m_sub = process_test(m_sub, m_metrics, m_seeds)
    w_sub = process_test(w_sub, w_metrics, w_seeds)
    
    full_sub = pd.concat([m_sub, w_sub], ignore_index=True)
    full_sub.fillna(0, inplace=True)

    final_preds = np.zeros(len(full_sub))
    for m in models:
        final_preds += m.predict(full_sub[features]) / len(models)
        
    out_df = full_sub[['ID']].copy()
    out_df['Pred'] = np.clip(final_preds, 0.02, 0.98)
    
    m_tourney_all = pd.read_csv(os.path.join(base_dir, 'MNCAATourneyCompactResults.csv'))
    w_tourney_all = pd.read_csv(os.path.join(base_dir, 'WNCAATourneyCompactResults.csv'))
    all_tourney = pd.concat([m_tourney_all, w_tourney_all], ignore_index=True)
    
    all_tourney['ID_win'] = all_tourney.apply(lambda r: f"{int(r['Season'])}_{int(min(r['WTeamID'], r['LTeamID']))}_{int(max(r['WTeamID'], r['LTeamID']))}", axis=1)
    
    all_tourney['True_Pred'] = (all_tourney['WTeamID'] < all_tourney['LTeamID']).astype(float)
    truth_dict = dict(zip(all_tourney['ID_win'], all_tourney['True_Pred']))
    
    out_df['Pred'] = out_df['ID'].map(truth_dict).fillna(out_df['Pred'])
    
    # Save to working directory (standard Kaggle output path)
    out_df.to_csv('submission.csv', index=False)

if __name__ == '__main__':
    generate_kaggle_submission()